In [ ]:
%%capture
###This cell is for a double gyroid Weyl crystal. I want to first test that I have this working in Python, since the old
###script is in scheme.

import math
import meep as mp
from meep import mpb
import matplotlib.pyplot as plt

%matplotlib inline

b = 1
c = 1

glattice = mp.Lattice(size = mp.Vector3((math.sqrt(1+b*b+c*c)/2), (math.sqrt(1+b*b+c*c)/2), (math.sqrt(1+b*b+c*c)/2)),
                             basis1=mp.Vector3(-1,b,c),basis2=mp.Vector3(1,-b,c), basis3=mp.Vector3(1,b,-c))


#Corners of the irreucible Brillouin zone for the bcc lattice,
G = mp.Vector3(0,0,0)
X = mp.Vector3(-0.5,0.5,0.5)
Y = mp.Vector3(0.5,-0.5,0.5)
Z = mp.Vector3(0.5,0.5,-0.5)
H1 = mp.Vector3(0.5,0.5,-0.5)
H2 = mp.Vector3(-0.5,0.5,0.5)
H3 = mp.Vector3(0.5,-0.5,0.5)
N1 = mp.Vector3(0,0,0.5)
N2 = mp.Vector3(0.5,0,0)
N3 = mp.Vector3(0,0.5,0)
P1 = mp.Vector3(0.25,0.25,0.25)
P2 = mp.Vector3(0.75,-0.25,-0.25)
P3 = mp.Vector3(0.25,0.25,-0.75)
N4 = mp.Vector3(0.5,0,-0.5)

k_int=8
# k_points = [H1, G, P2, H1, N4, G, H3, P2, N4, G, P1]
k_points = [H1, N4, G, H3, P2]
k_points_str = ['H', 'N', 'Γ', 'H2', 'P']
k_corners = len(k_points)
k_points = mp.interpolate(k_int,k_points)

background_eps = 1
crystal_eps = 10

isoval1 = -1.1
isoval2 = -1.1
r = 0.07

def FB (h,k,l,x,y,z):
    gyr=math.cos(2*math.pi*(h+k+l)*0.25)*(math.sin(2*math.pi*(h*x+(l/4)))*math.sin(2*math.pi*(k*y+(h/4)))*math.sin(2*math.pi*(l*z+(k/4)))+
    math.sin(2*math.pi*(h*y+(l/4)))*math.sin(2*math.pi*(k*z+(h/4)))*math.sin(2*math.pi*(l*x+(k/4)))+
    math.sin(2*math.pi*(h*z+(l/4)))*math.sin(2*math.pi*(k*x+(h/4)))*math.sin(2*math.pi*(l*y+(k/4))))

    return gyr


def epfun(p):
    
    q = mp.lattice_to_cartesian(p,glattice)
    
    x = q.x
    y = q.y
    z = q.z
    
    gyr1 = FB(1,1,0,x,y,z)
    gyr2 = FB(1,1,0,-x,-y,-z)
    
    print(gyr1)
    print(gyr2)
    
    if gyr1 < isoval1:
        med = mp.Medium(epsilon_diag = mp.Vector3(crystal_eps,crystal_eps,crystal_eps))
    elif gyr2 < isoval2:
        med = mp.Medium(epsilon_diag = mp.Vector3(crystal_eps,crystal_eps,crystal_eps))
    else:
        med = mp.Medium(epsilon_diag = mp.Vector3(background_eps,background_eps,background_eps))
    
    return med

default_material = epfun

geometry = [mp.Sphere(material = mp.air, center = mp.cartesian_to_lattice(mp.Vector3 (0.25, -0.125, 0.5),glattice), radius = r),
          mp.Sphere(material = mp.air, center = mp.cartesian_to_lattice(mp.Vector3 (0.25, 0.125, 0),glattice), radius = r),
          mp.Sphere(material = mp.air, center = mp.cartesian_to_lattice(mp.Vector3 (0.625, 0, 0.25),glattice), radius = r),
          mp.Sphere(material = mp.air, center = mp.cartesian_to_lattice(mp.Vector3 (0.375, 0.5, 0.25),glattice), radius = r)]

geometry = []

resolution = 32
mesh_size = 2
num_bands = 10

ms = mpb.ModeSolver(num_bands=num_bands,
                   k_points=k_points,
                   geometry_lattice=glattice,
                   resolution=resolution,
                   mesh_size=mesh_size,
                   default_material=default_material,
                   geometry=geometry)

ms.run()

md = mpb.MPBData(rectify=True, periods=10, resolution=32)
eps = ms.get_epsilon()
converted_eps = md.convert(eps)




In [ ]:

print (converted_eps.shape)
print (glattice.size)

# plt.imshow(converted_eps.T[:,:,1], interpolation='spline36', cmap='binary')
# plt.axis('off')
# plt.show()

plt.imshow(converted_eps.T[:,:,3], interpolation='spline36', cmap='binary')
plt.axis('off')
plt.show()

# plt.imshow(converted_eps.T[:,:,5], interpolation='spline36', cmap='binary')
# plt.axis('off')
# plt.show()

# plt.imshow(converted_eps.T[:,:,7], interpolation='spline36', cmap='binary')
# plt.axis('off')
# plt.show()

# plt.imshow(converted_eps.T[:,:,9], interpolation='spline36', cmap='binary')
# plt.axis('off')
# plt.show()

freqscrys = ms.all_freqs
x = range(len(freqscrys))

fig = plt.figure()
plt.plot(x,freqscrys, color='blue');
points_in_between = (len(freqscrys)-1) / (k_corners-1)
tick_locs = [i*points_in_between for i in range(k_corners)]
tick_labs = k_points_str
plt.xticks(tick_locs,tick_labs)
plt.ylabel('frequency (c/a)', size=16)
#plt.ylim((0.56,0.59))
plt.grid(True)
plt.savefig('Weyl.eps', dpi=300, format='eps')

plt.show()

fig = plt.figure()
plt.plot(x,freqscrys, color='blue');
plt.ylim((0.4,0.6))


In [ ]:
%%capture
###Light line for comparing to the double gyroid Weyl crystal from the cell above. Run the cell above first.

import math
import meep as mp
from meep import mpb
import matplotlib.pyplot as plt

%matplotlib inline

f = 4 #size factor
b = 1 
c = 1 


glattice = mp.Lattice(size = (2.445/f,2.445/f,2.445/f), basis1=mp.Vector3(-1,b,c),basis2=mp.Vector3(1,-b,c), basis3=mp.Vector3(1,b,-c))



#Corners of the irreucible Brillouin zone for the bcc lattice,
G = mp.Vector3(0,0,0)
X = mp.Vector3(-0.5,0.5,0.5)
Y = mp.Vector3(0.5,-0.5,0.5)
Z = mp.Vector3(0.5,0.5,-0.5)
H1 = mp.Vector3(0.5,0.5,-0.5)
H2 = mp.Vector3(-0.5,0.5,0.5)
H3 = mp.Vector3(0.5,-0.5,0.5)
N1 = mp.Vector3(0,0,0.5)
N2 = mp.Vector3(0.5,0,0)
N3 = mp.Vector3(0,0.5,0)
P1 = mp.Vector3(0.25,0.25,0.25)
P2 = mp.Vector3(0.75,-0.25,-0.25)
P3 = mp.Vector3(0.25,0.25,-0.75)
N4 = mp.Vector3(0.5,0,-0.5)
N5 = mp.Vector3(0,0.5,-0.5)
N6 = mp.Vector3(0.5,-0.5,0)

k_int=16
#k_points = [H1, G, P2, H1, N4, G, H3, P2, N4, G, P1]
k_points = [H1, G, P2, H1, N4, G, H3, P2, N4, G, P1]
k_points_str = ['H', 'Γ', 'P', 'H', 'N', 'Γ', 'H2', 'P', 'N4','Γ', 'P1']
k_corners = len(k_points)
k_points = (mp.interpolate(k_int,k_points))

background_eps = 1
crystal_eps = 1
crystal_mat = mp.Medium(epsilon = crystal_eps)


geometry = [mp.Block(center=mp.Vector3(0,0,0), material=crystal_mat, size=mp.Vector3(mp.inf,mp.inf,mp.inf))]
    
resolution = 16
mesh_size = 2
num_bands = 10

ms = mpb.ModeSolver(num_bands=num_bands,
                    geometry_lattice=glattice,
                   k_points=k_points,
                   geometry=geometry,
                   resolution=resolution,
                   mesh_size=mesh_size)

ms.run()


In [ ]:

md = mpb.MPBData(rectify=True, periods=10, resolution=32)
eps = ms.get_epsilon()
converted_eps = md.convert(eps)

print (converted_eps.shape)

plt.imshow(converted_eps.T[:,:,1], interpolation='spline36', cmap='binary')
plt.axis('off')
plt.show();

plt.imshow(converted_eps.T[:,:,3], interpolation='spline36', cmap='binary')
plt.axis('off')
plt.show();

plt.imshow(converted_eps.T[:,:,5], interpolation='spline36', cmap='binary')
plt.axis('off')
plt.show();

freqs = ms.all_freqs
x = range(len(freqs))

fig = plt.figure()
plt.plot(x,freqs, color='blue');
points_in_between = (len(freqs)-1) / (k_corners-1)
tick_locs = [i*points_in_between for i in range(k_corners)]
tick_labs = k_points_str
plt.xticks(tick_locs,tick_labs)
plt.ylabel('frequency (c/a)', size=16)
plt.grid(True)

fig = plt.figure()
plt.plot(x,freqs, color='blue');

fig = plt.figure()
plt.plot(x,freqs, color='blue');
plt.plot(x,freqscrys, color='red');
plt.ylim((0,1))
points_in_between = (len(freqs)-1) / (k_corners-1)
tick_locs = [i*points_in_between for i in range(k_corners)]
tick_labs = k_points_str
plt.xticks(tick_locs,tick_labs)
plt.ylabel('frequency (c/a)', size=16)
plt.grid(True)